# Real Data Experiment: Year Prediction MSD Dataset

Compares 7 regression methods on the Million Song Dataset (Year Prediction subset):
- RF + Ridge (Random Fourier Features + Ridge Regression)
- DNN (Deep Neural Network)
- ResNet (Residual Network)
- DKL (Deep Kernel Learning)
- DGP (Deep Gaussian Process)
- MLKM (Multi-Layer Kernel Machine)
- ResKernelNet (Residual Kernel Machine)

Metrics: Train MSE, Test MSE, Runtime, CPU/GPU Memory, Conformal Prediction Coverage & Interval Length

**v2 fixes vs original MSD notebook:**
- Fixed `rkn_result` unpacking (8 values, not 5)
- Fixed conformal prediction calls (use `conformal_prediction_split_batch` with correct signature)
- Fixed `run_mlkm` missing device assignment
- Fixed `run_reskernelnet` seed inconsistency
- Batched conformal prediction for large test set (46K samples)

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import gpytorch
from gpytorch.models import ExactGP
from gpytorch.means import ConstantMean
from gpytorch.kernels import RBFKernel, ScaleKernel
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.mlls import ExactMarginalLogLikelihood
from gpytorch.distributions import MultivariateNormal
from gpytorch import variational

import psutil, os, sys, resource, gc, multiprocessing, threading, time, random
from tqdm import tqdm

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

PyTorch version: 2.9.1
CUDA available: False


In [2]:
# ── Data Loading: Year Prediction MSD ────────────────────────────────────────
# Official train/test split:
#   First 463715 rows → training pool; last 51630 rows → test
# We use a smaller subset for feasibility:
#   Take the first 10% of all rows, split 50/50 into train & calibration.
#   Use the next 90% as the test set.

data_path = '../../data/YearPredictionMSD/YearPredictionMSD.txt'

df = pd.read_csv(data_path, header=None)
print('Full dataset shape:', df.shape)   # (515345, 91)

# Take 10% subset
n_total  = len(df)
n_subset = int(n_total * 0.10)   # 51534
df_sub   = df.iloc[:n_subset].reset_index(drop=True)
print('Working subset shape:', df_sub.shape)

# Target = column 0 (year), Features = columns 1-90
X_all = df_sub.iloc[:, 1:].values.astype(np.float32)   # (51534, 90)
y_all = df_sub.iloc[:, 0 ].values.astype(np.float32)   # (51534,)

# Split: first 10% of subset → train + calibration ; rest → test
n_train_cal = int(n_subset * 0.10)   # 5153
X_train_cal, y_train_cal = X_all[:n_train_cal], y_all[:n_train_cal]
X_test,      y_test      = X_all[n_train_cal:], y_all[n_train_cal:]

# Train / Calibration 50-50 split
n_half = n_train_cal // 2
X_train, y_train = X_train_cal[:n_half], y_train_cal[:n_half]
X_cal,   y_cal   = X_train_cal[n_half:], y_train_cal[n_half:]

# Normalise (fit on train only)
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_cal   = scaler.transform(X_cal)
X_test  = scaler.transform(X_test)

# Sanity checks
for name, X, y in [('Train', X_train, y_train),
                    ('Calib', X_cal,   y_cal),
                    ('Test',  X_test,  y_test)]:
    has_nan = np.isnan(X).any() or np.isnan(y).any()
    has_inf = np.isinf(X).any() or np.isinf(y).any()
    print(f'{name}: X={X.shape}, y={y.shape}, NaN={has_nan}, Inf={has_inf}')

# Convert to numpy float32 for consistent downstream usage
train_x, train_y = X_train.astype(np.float32), y_train.astype(np.float32)
calibration_x, calibration_y = X_cal.astype(np.float32), y_cal.astype(np.float32)
test_x,  test_y  = X_test.astype(np.float32),  y_test.astype(np.float32)

Full dataset shape: (515345, 91)
Working subset shape: (51534, 91)
Train: X=(2576, 90), y=(2576,), NaN=False, Inf=False
Calib: X=(2577, 90), y=(2577,), NaN=False, Inf=False
Test: X=(46381, 90), y=(46381,), NaN=False, Inf=False


In [3]:
# ── Global hyperparameters ────────────────────────────────────────────────────
INPUT_DIM   = 90    # number of features in MSD
HIDDEN_DIM1 = 256
HIDDEN_DIM2 = 128
HIDDEN_DIM3 = 64

BATCH_SIZE        = 256
MAX_EPOCHS        = 1000
EARLY_STOP_WINDOW = 50
LR                = 1e-4
MOMENTUM          = 0.95
WEIGHT_DECAY      = 1e-3
SEED              = 7199

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

Using device: cpu


In [4]:
# ── Dataset helper ────────────────────────────────────────────────────────────
class mydataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __len__(self):
        return len(self.x)
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [5]:
# ── Reproducibility & measurement utilities ───────────────────────────────────
def set_global_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def measure_time(func, *args, **kwargs):
    """Wall-clock time of func(*args, **kwargs). Returns (result, seconds)."""
    t0 = time.time()
    result = func(*args, **kwargs)
    return result, time.time() - t0


def measure_memory_cpu(func, *args, **kwargs):
    """CPU RSS memory during func(). Returns (result, delta_MB, peak_MB)."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    process   = psutil.Process(os.getpid())
    mem_before = process.memory_info().rss
    peak_rss   = [mem_before]
    stop_evt   = threading.Event()

    def _monitor():
        while not stop_evt.is_set():
            try:
                cur = process.memory_info().rss
                if cur > peak_rss[0]:
                    peak_rss[0] = cur
            except Exception:
                pass
            stop_evt.wait(0.05)

    t = threading.Thread(target=_monitor, daemon=True)
    t.start()
    result    = func(*args, **kwargs)
    mem_after = process.memory_info().rss
    stop_evt.set()
    t.join()
    if mem_after > peak_rss[0]:
        peak_rss[0] = mem_after

    delta_mb = (mem_after  - mem_before) / 1024**2
    peak_mb  = (peak_rss[0] - mem_before) / 1024**2
    return result, delta_mb, peak_mb


def measure_memory_gpu(func, *args, **kwargs):
    """Peak GPU memory during func(). Returns (result, peak_MB or None)."""
    if not torch.cuda.is_available():
        return func(*args, **kwargs), None
    torch.cuda.reset_peak_memory_stats()
    result    = func(*args, **kwargs)
    peak_mb   = torch.cuda.max_memory_allocated() / 1024**2
    return result, peak_mb

In [6]:
# ── Conformal Prediction (batched for large test sets) ────────────────────────
def conformal_prediction_split_batch(
    net, device,
    calibration_x, calibration_y,
    test_x, test_y,
    alpha=0.05,
    batch_size=1024
):
    """
    Split conformal prediction for regression (batched for efficiency).

    Scores: R_i = |y_i - f(x_i)|  for calibration points.
    Quantile: q_hat = ceil((m+1)(1-alpha))-th smallest score.
    Intervals: [f(x) - q_hat, f(x) + q_hat].

    Returns
    -------
    coverage : float
    avg_interval_length : float   (= 2 * q_hat for homoskedastic)
    q_hat : float
    """
    net.eval()

    # ── Step 1: Calibration scores (batched) ─────────────────────────────────
    cal_preds = []
    with torch.no_grad():
        for i in range(0, len(calibration_x), batch_size):
            xb = torch.from_numpy(calibration_x[i:i+batch_size]).float().to(device)
            cal_preds.append(net(xb).squeeze().cpu().numpy())
    cal_preds = np.concatenate(cal_preds)
    scores = np.abs(calibration_y - cal_preds)

    # ── Step 2: Conformal quantile ────────────────────────────────────────────
    m = len(scores)
    sorted_scores = np.sort(scores)
    k = int(np.ceil((m + 1) * (1 - alpha)))
    k = min(max(k, 1), m)
    q_hat = float(sorted_scores[k - 1])

    # ── Step 3: Test coverage (batched) ──────────────────────────────────────
    test_preds = []
    with torch.no_grad():
        for i in range(0, len(test_x), batch_size):
            xb = torch.from_numpy(test_x[i:i+batch_size]).float().to(device)
            test_preds.append(net(xb).squeeze().cpu().numpy())
    test_preds = np.concatenate(test_preds)

    lower = test_preds - q_hat
    upper = test_preds + q_hat
    coverage = float(np.mean((lower <= test_y) & (test_y <= upper)))
    avg_interval_length = 2.0 * q_hat

    return coverage, avg_interval_length, q_hat

In [7]:
# ── Random Fourier Feature ────────────────────────────────────────────────────
def _sample_1d(pdf, gamma):
    if pdf == 'G':
        return torch.randn(1) * gamma
    elif pdf == 'L':
        return torch.distributions.Laplace(torch.tensor([0.0]), torch.tensor([1.0])).sample() * gamma
    elif pdf == 'C':
        return torch.distributions.Cauchy(torch.tensor([0.0]), torch.tensor([1.0])).sample() * gamma

def _sample(pdf, gamma, d):
    return torch.tensor([_sample_1d(pdf, gamma) for _ in range(d)])

class RandomFourierFeature:
    """Random Fourier Feature mapping x (n,d) → phi(x) (n,D)."""
    def __init__(self, d, D, kernel='G', gamma=1.0):
        self.d, self.D, self.gamma = d, D, gamma
        kernel = kernel.upper()
        if kernel not in ('G', 'L', 'C'):
            raise ValueError('kernel must be G, L, or C')
        self.kernel = kernel
        self.b = torch.rand(D) * 2 * torch.pi
        self.W = _sample(kernel, gamma, d * D).reshape(D, d)

    def transform(self, x):
        # x: (n, d)  →  result: (n, D)
        ones = torch.ones(len(x), device=x.device).reshape(1, -1)   # (1, n)
        W = self.W.to(x.device)
        b = self.b.to(x.device)
        result = torch.sqrt(torch.tensor(2.0 / self.D, device=x.device)) * \
                 torch.cos(W @ x.T + b.reshape(-1, 1) * ones)
        return result.T

In [8]:
# ── Weight initialisation ─────────────────────────────────────────────────────
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.uniform_(m.weight, -0.1, 0.1)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

In [9]:
# ── DNN ───────────────────────────────────────────────────────────────────────
class Net(nn.Module):
    def __init__(self, h1=HIDDEN_DIM1, h2=HIDDEN_DIM2, h3=HIDDEN_DIM3):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(INPUT_DIM, h1), nn.ReLU(),
            nn.Linear(h1, h2),        nn.ReLU(),
            nn.Linear(h2, h3),        nn.ReLU(),
            nn.Linear(h3, 1)
        )
    def forward(self, x):
        return self.layers(x)


def run_dnn(train_x, train_y, test_x, test_y,
            h1=HIDDEN_DIM1, h2=HIDDEN_DIM2, h3=HIDDEN_DIM3,
            batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
            early_stop_window=EARLY_STOP_WINDOW,
            lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
            device=None, verbose=False, seed=SEED):
    set_global_seed(seed)
    if device is None:
        device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    nntrain_x = torch.from_numpy(train_x).float()
    nntrain_y = torch.from_numpy(train_y).float().squeeze()
    train_loader = DataLoader(mydataset(nntrain_x, nntrain_y),
                              batch_size=batch_size, shuffle=True)

    net = Net(h1, h2, h3).to(device)
    net.apply(init_weights)
    criterion = nn.MSELoss()
    optimizer = optim.SGD(net.parameters(), lr=lr,
                          momentum=momentum, weight_decay=weight_decay)

    trainloss, testloss = [], []
    for epoch in range(max_epochs):
        net.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(net(x).squeeze(), y)
            loss.backward()
            optimizer.step()

        net.eval()
        with torch.no_grad():
            p_tr = net(torch.from_numpy(train_x).float().to(device)).squeeze().cpu().numpy()
            trainloss.append(mean_squared_error(train_y, p_tr))
            p_te = net(torch.from_numpy(test_x).float().to(device)).squeeze().cpu().numpy()
            testloss.append(mean_squared_error(test_y, p_te))

        if epoch > early_stop_window:
            if trainloss[-1] > max(trainloss[-early_stop_window:-1]):
                if verbose: print(f'DNN early stop @ epoch {epoch}')
                break
        if verbose and epoch % 100 == 0:
            print(f'epoch {epoch:4d}  train={trainloss[-1]:.4f}  test={testloss[-1]:.4f}')

    return trainloss[-1], testloss[-1], len(trainloss), net, optimizer, trainloss, testloss, device

In [10]:
# ── ResNet ────────────────────────────────────────────────────────────────────
class ResidualBlock(nn.Module):
    """Parallel-branch residual block: ReLU(fc1(x) + fc2(x))."""
    def __init__(self, in_f, out_f):
        super().__init__()
        self.fc1 = nn.Linear(in_f, out_f)
        self.fc2 = nn.Linear(in_f, out_f)
    def forward(self, x):
        return F.relu(F.relu(self.fc1(x)) + self.fc2(x))


class ResNet(nn.Module):
    def __init__(self, h1=HIDDEN_DIM1, h2=HIDDEN_DIM2, h3=HIDDEN_DIM3):
        super().__init__()
        self.blocks = nn.Sequential(
            ResidualBlock(INPUT_DIM, h1),
            ResidualBlock(h1, h2),
            ResidualBlock(h2, h3)
        )
        self.out = nn.Linear(h3, 1)
    def forward(self, x):
        return self.out(self.blocks(x))


def run_resnet(train_x, train_y, test_x, test_y,
               h1=HIDDEN_DIM1, h2=HIDDEN_DIM2, h3=HIDDEN_DIM3,
               batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
               early_stop_window=EARLY_STOP_WINDOW,
               lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
               device=None, verbose=False, seed=SEED):
    set_global_seed(seed)
    if device is None:
        device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    nntrain_x = torch.from_numpy(train_x).float()
    nntrain_y = torch.from_numpy(train_y).float().squeeze()
    train_loader = DataLoader(mydataset(nntrain_x, nntrain_y),
                              batch_size=batch_size, shuffle=True)

    net = ResNet(h1, h2, h3).to(device)
    net.apply(init_weights)
    criterion = nn.MSELoss()
    optimizer = optim.SGD(net.parameters(), lr=lr,
                          momentum=momentum, weight_decay=weight_decay)

    trainloss, testloss = [], []
    for epoch in range(max_epochs):
        net.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(net(x).squeeze(), y)
            loss.backward()
            optimizer.step()

        net.eval()
        with torch.no_grad():
            p_tr = net(torch.from_numpy(train_x).float().to(device)).squeeze().cpu().numpy()
            trainloss.append(mean_squared_error(train_y, p_tr))
            p_te = net(torch.from_numpy(test_x).float().to(device)).squeeze().cpu().numpy()
            testloss.append(mean_squared_error(test_y, p_te))

        if epoch > early_stop_window:
            if trainloss[-1] > max(trainloss[-early_stop_window:-1]):
                if verbose: print(f'ResNet early stop @ epoch {epoch}')
                break
        if verbose and epoch % 100 == 0:
            print(f'epoch {epoch:4d}  train={trainloss[-1]:.4f}  test={testloss[-1]:.4f}')

    return trainloss[-1], testloss[-1], len(trainloss), net, optimizer, trainloss, testloss, device

In [11]:
# ── Deep Kernel Learning (DKL) ────────────────────────────────────────────────
class DKLFeatureExtractor(nn.Module):
    def __init__(self, in_dim=INPUT_DIM, h1=HIDDEN_DIM1, h2=HIDDEN_DIM2,
                 h3=HIDDEN_DIM3, out_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, h1), nn.ReLU(),
            nn.Linear(h1, h2),    nn.ReLU(),
            nn.Linear(h2, h3),    nn.ReLU(),
            nn.Linear(h3, out_dim)
        )
    def forward(self, x):
        return self.net(x)


class DKLExactGP(ExactGP):
    def __init__(self, train_x, train_y, likelihood, feature_extractor):
        super().__init__(train_x, train_y, likelihood)
        self.feature_extractor = feature_extractor
        self.mean_module  = ConstantMean()
        self.covar_module = ScaleKernel(RBFKernel())

    def forward(self, x):
        feat = self.feature_extractor(x)
        mean  = self.mean_module(feat)
        covar = self.covar_module(feat)
        return MultivariateNormal(mean, covar)


def run_dkl(train_x, train_y, test_x, test_y,
            h1=HIDDEN_DIM1, h2=HIDDEN_DIM2, h3=HIDDEN_DIM3,
            training_iter=300, seed=SEED):
    """Deep Kernel Learning. Returns (train_mse, test_mse)."""
    set_global_seed(seed)
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    train_x_t = torch.from_numpy(train_x).float().to(device)
    train_y_t = torch.from_numpy(train_y).float().squeeze().to(device)
    test_x_t  = torch.from_numpy(test_x).float().to(device)

    feat       = DKLFeatureExtractor(INPUT_DIM, h1, h2, h3).to(device)
    likelihood = GaussianLikelihood().to(device)
    model      = DKLExactGP(train_x_t, train_y_t, likelihood, feat).to(device)

    model.train(); likelihood.train()
    optimizer = torch.optim.Adam([
        {'params': model.feature_extractor.parameters(), 'lr': 1e-3},
        {'params': model.covar_module.parameters()},
        {'params': model.mean_module.parameters()},
        {'params': likelihood.parameters()},
    ], lr=0.05)
    mll = ExactMarginalLogLikelihood(likelihood, model)

    for _ in range(training_iter):
        optimizer.zero_grad()
        loss = -mll(model(train_x_t), train_y_t)
        loss.backward()
        optimizer.step()

    model.eval(); likelihood.eval()
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        pred_tr = likelihood(model(train_x_t)).mean
        pred_te = likelihood(model(test_x_t)).mean

    train_mse = mean_squared_error(train_y, pred_tr.cpu().numpy())
    test_mse  = mean_squared_error(test_y,  pred_te.cpu().numpy())
    return train_mse, test_mse

In [12]:
# ── Deep Gaussian Process (DGP) ───────────────────────────────────────────────
class DGPLayer(gpytorch.models.deep_gps.DeepGPLayer):
    def __init__(self, input_dims, output_dims, inducing_points):
        if output_dims is not None:
            batch_shape = torch.Size([output_dims])
        else:
            batch_shape = torch.Size([])

        var_strat = variational.CholeskyVariationalDistribution(
            inducing_points.size(-2), batch_shape=batch_shape)
        var_strat = variational.VariationalStrategy(
            self, inducing_points, var_strat, learn_inducing_locations=True)

        super().__init__(var_strat, input_dims, output_dims)
        self.mean_module  = ConstantMean(batch_shape=batch_shape)
        self.covar_module = ScaleKernel(
            RBFKernel(batch_shape=batch_shape, ard_num_dims=input_dims),
            batch_shape=batch_shape
        )

    def forward(self, x):
        return MultivariateNormal(self.mean_module(x), self.covar_module(x))


def run_dgp(train_x, train_y, test_x, test_y,
            training_iter=1000, batch_size=256,
            num_inducing=128, hidden_dim=5, seed=SEED):
    """Deep Gaussian Process. Returns (train_mse, test_mse)."""
    set_global_seed(seed)
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    train_x_t = torch.from_numpy(train_x).float().to(device)
    train_y_t = torch.from_numpy(train_y).float().squeeze().to(device)
    test_x_t  = torch.from_numpy(test_x).float().to(device)

    class TwoLayerDGP(gpytorch.models.deep_gps.DeepGP):
        def __init__(self):
            super().__init__()
            ind1 = train_x_t[torch.randperm(train_x_t.size(0))[:num_inducing]]
            ind2 = torch.randn(num_inducing, hidden_dim).to(device)
            self.hidden_layer = DGPLayer(train_x_t.shape[1], hidden_dim, ind1)
            self.output_layer = DGPLayer(hidden_dim, None, ind2)
            self.likelihood   = GaussianLikelihood().to(device)

        def forward(self, x):
            return self.output_layer(self.hidden_layer(x))

        def predict(self, x):
            self.eval(); self.likelihood.eval()
            with torch.no_grad(), gpytorch.settings.num_likelihood_samples(16):
                return self.likelihood(self(x)).mean.mean(dim=0)

    dgp = TwoLayerDGP().to(device)
    dgp.train(); dgp.likelihood.train()
    optimizer = torch.optim.Adam(dgp.parameters(), lr=0.01)
    elbo = gpytorch.mlls.DeepApproximateMLL(
        gpytorch.mlls.VariationalELBO(
            dgp.likelihood, dgp, num_data=train_x_t.size(0)))

    for _ in range(training_iter):
        perm = torch.randperm(train_x_t.size(0), device=device)
        for j in range(0, train_x_t.size(0), batch_size):
            idx = perm[j:j+batch_size]
            optimizer.zero_grad()
            loss = -elbo(dgp(train_x_t[idx]), train_y_t[idx])
            loss.backward()
            optimizer.step()

    train_mse = mean_squared_error(train_y, dgp.predict(train_x_t).cpu().numpy())
    test_mse  = mean_squared_error(test_y,  dgp.predict(test_x_t).cpu().numpy())
    return train_mse, test_mse

In [13]:
# ── RF + Ridge ────────────────────────────────────────────────────────────────
def run_rf_ridge(train_x, train_y, test_x, test_y,
                 D=500, gamma=0.4, kernel='G', seed=SEED):
    """Random Fourier Features + Ridge regression. Returns (train_mse, test_mse)."""
    set_global_seed(seed)
    rff = RandomFourierFeature(train_x.shape[1], D, kernel=kernel, gamma=gamma)

    x_tr = torch.from_numpy(train_x).float()
    x_te = torch.from_numpy(test_x).float()
    with torch.no_grad():
        phi_tr = rff.transform(x_tr).numpy()
        phi_te = rff.transform(x_te).numpy()

    ridge = RidgeCV(alphas=[1e-4, 1e-3, 1e-2, 1e-1, 1.0],
                    cv=KFold(n_splits=5, shuffle=True, random_state=seed))
    ridge.fit(phi_tr, train_y)

    train_mse = mean_squared_error(train_y, ridge.predict(phi_tr))
    test_mse  = mean_squared_error(test_y,  ridge.predict(phi_te))
    return train_mse, test_mse

In [14]:
# ── MLKM (Multi-Layer Kernel Machine) ─────────────────────────────────────────
def run_mlkm(train_x, train_y, test_x, test_y,
             D1=500, D2=500, D3=500,
             h1=HIDDEN_DIM1, h2=HIDDEN_DIM2, h3=HIDDEN_DIM3,
             batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
             early_stop_window=EARLY_STOP_WINDOW,
             lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
             device=None, verbose=False, seed=SEED):
    """
    MLKM: each layer applies RFF then a learned linear map.
    Architecture: INPUT --rff1--> D1 --fc1--> h1 --rff2--> D2 --fc2--> h2
                       --rff3--> D3 --fc3--> h3 --out--> 1
    """
    set_global_seed(seed)
    # FIX: always assign device (was missing in original)
    if device is None:
        device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    nntrain_x = torch.from_numpy(train_x).float()
    nntrain_y = torch.from_numpy(train_y).float().squeeze()
    train_loader = DataLoader(mydataset(nntrain_x, nntrain_y),
                              batch_size=batch_size, shuffle=True)

    rff1 = RandomFourierFeature(INPUT_DIM, D1, kernel='G', gamma=0.1)
    rff2 = RandomFourierFeature(h1, D2, kernel='G', gamma=0.4)
    rff3 = RandomFourierFeature(h2, D3, kernel='G', gamma=0.4)

    class KernelNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(D1, h1)
            self.fc2 = nn.Linear(D2, h2)
            self.fc3 = nn.Linear(D3, h3)
            self.out = nn.Linear(h3, 1)
        def forward(self, x):
            x = self.fc1(rff1.transform(x))
            x = self.fc2(rff2.transform(x))
            x = self.fc3(rff3.transform(x))
            return self.out(x)

    net = KernelNet().to(device)
    net.apply(init_weights)
    criterion = nn.MSELoss()
    optimizer = optim.SGD(net.parameters(), lr=lr,
                          momentum=momentum, weight_decay=weight_decay)

    trainloss, testloss = [], []
    for epoch in range(max_epochs):
        net.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(net(x).squeeze(), y)
            loss.backward()
            optimizer.step()

        net.eval()
        with torch.no_grad():
            p_tr = net(torch.from_numpy(train_x).float().to(device)).squeeze().cpu().numpy()
            trainloss.append(mean_squared_error(train_y, p_tr))
            p_te = net(torch.from_numpy(test_x).float().to(device)).squeeze().cpu().numpy()
            testloss.append(mean_squared_error(test_y, p_te))

        if epoch > early_stop_window:
            if trainloss[-1] > max(trainloss[-early_stop_window:-1]):
                if verbose: print(f'MLKM early stop @ epoch {epoch}')
                break
        if verbose and epoch % 100 == 0:
            print(f'epoch {epoch:4d}  train={trainloss[-1]:.4f}  test={testloss[-1]:.4f}')

    return trainloss[-1], testloss[-1], len(trainloss), net, optimizer, trainloss, testloss, device

In [15]:
# ── ResKernelNet (Residual Kernel Machine) ─────────────────────────────────────
def run_reskernelnet(train_x, train_y, test_x, test_y,
                     D1=500,
                     h1=HIDDEN_DIM1, h2=HIDDEN_DIM2, h3=HIDDEN_DIM3,
                     batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
                     early_stop_window=EARLY_STOP_WINDOW,
                     lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
                     device=None, verbose=False, seed=SEED):
    """
    ResKernelNet:
      block_l(x) = A_l x + Delta_l x + W_l phi_l(x)
    Architecture:
      INPUT --block1(D1,h1)--> h1 --block2(D2=h1,h2)--> h2
            --block3(D3=h2,h3)--> h3 --out--> 1
    Note: D2=h1 and D3=h2 so that RFF dimensions match hidden dims.
    """
    # FIX: use consistent seed (original used torch.manual_seed(1))
    set_global_seed(seed)
    if device is None:
        device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    D2 = h1   # must match for residual connections
    D3 = h2

    nntrain_x = torch.from_numpy(train_x).float()
    nntrain_y = torch.from_numpy(train_y).float().squeeze()
    train_loader = DataLoader(mydataset(nntrain_x, nntrain_y),
                              batch_size=batch_size, shuffle=True)

    rff1 = RandomFourierFeature(INPUT_DIM, D1, kernel='G', gamma=0.1)
    rff2 = RandomFourierFeature(h1, D2, kernel='G', gamma=0.4)
    rff3 = RandomFourierFeature(h2, D3, kernel='G', gamma=0.4)

    class ResKernelBlock(nn.Module):
        def __init__(self, d_in, d_phi, d_out, rff):
            super().__init__()
            self.rff = rff
            self.W   = nn.Linear(d_phi, d_out)
            self.A   = nn.Identity() if d_in == d_out else nn.Linear(d_in, d_out, bias=False)
            self.Delta = nn.Linear(d_in, d_out, bias=False)
            nn.init.zeros_(self.Delta.weight)
        def forward(self, x):
            return self.A(x) + self.Delta(x) + self.W(self.rff.transform(x))

    class ResKernelNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.b1  = ResKernelBlock(INPUT_DIM, D1, h1, rff1)
            self.b2  = ResKernelBlock(h1, D2, h2, rff2)
            self.b3  = ResKernelBlock(h2, D3, h3, rff3)
            self.out = nn.Linear(h3, 1)
        def forward(self, x):
            return self.out(self.b3(self.b2(self.b1(x))))

    net = ResKernelNet().to(device)
    criterion = nn.MSELoss()
    optimizer = optim.SGD(net.parameters(), lr=lr,
                          momentum=momentum, weight_decay=weight_decay)

    trainloss, testloss = [], []
    for epoch in range(max_epochs):
        net.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(net(x).squeeze(), y)
            loss.backward()
            optimizer.step()

        net.eval()
        with torch.no_grad():
            p_tr = net(torch.from_numpy(train_x).float().to(device)).squeeze().cpu().numpy()
            trainloss.append(mean_squared_error(train_y, p_tr))
            p_te = net(torch.from_numpy(test_x).float().to(device)).squeeze().cpu().numpy()
            testloss.append(mean_squared_error(test_y, p_te))

        if epoch > early_stop_window:
            if trainloss[-1] > max(trainloss[-early_stop_window:-1]):
                if verbose: print(f'ResKernelNet early stop @ epoch {epoch}')
                break
        if verbose and epoch % 100 == 0:
            print(f'epoch {epoch:4d}  train={trainloss[-1]:.4f}  test={testloss[-1]:.4f}')

    return trainloss[-1], testloss[-1], len(trainloss), net, optimizer, trainloss, testloss, device

In [16]:
# ── Baseline run (default hidden dims) ───────────────────────────────────────
# RF + Ridge
def rf_ridge_core():
    return run_rf_ridge(train_x, train_y, test_x, test_y)

((rf_result, rf_gpu), rf_cpu_delta, rf_cpu_peak), rf_time = measure_time(
    lambda: measure_memory_cpu(lambda: measure_memory_gpu(rf_ridge_core)))
rf_train_mse, rf_test_mse = rf_result
print(f'RF+Ridge  train={rf_train_mse:.4f}  test={rf_test_mse:.4f}  '
      f'time={rf_time:.1f}s  cpu_peak={rf_cpu_peak:.1f}MB')

RF+Ridge  train=87.3664  test=126.3447  time=0.6s  cpu_peak=6.0MB


In [17]:
# DNN
def dnn_core():
    return run_dnn(train_x, train_y, test_x, test_y)

((dnn_result, dnn_gpu), dnn_cpu_delta, dnn_cpu_peak), dnn_time = measure_time(
    lambda: measure_memory_cpu(lambda: measure_memory_gpu(dnn_core)))
dnn_train_mse, dnn_test_mse, dnn_epochs, dnn_net, dnn_optimizer, \
    dnn_trainloss, dnn_testloss, dnn_device = dnn_result
print(f'DNN  train={dnn_train_mse:.4f}  test={dnn_test_mse:.4f}  '
      f'epochs={dnn_epochs}  time={dnn_time:.1f}s  cpu_peak={dnn_cpu_peak:.1f}MB')

/Users/wanghd/Desktop/Research/Dai/Multi_layer_kernel_machines/Multi-Layer-Kernel-Machine/venv/lib/python3.13/site-packages/numpy/_core/_methods.py:134: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
/Users/wanghd/Desktop/Research/Dai/Multi_layer_kernel_machines/Multi-Layer-Kernel-Machine/venv/lib/python3.13/site-packages/numpy/_core/_methods.py:134: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
/Users/wanghd/Desktop/Research/Dai/Multi_layer_kernel_machines/Multi-Layer-Kernel-Machine/venv/lib/python3.13/site-packages/numpy/_core/_methods.py:134: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
/Users/wanghd/Desktop/Research/Dai/Multi_layer_kernel_machines/Multi-Layer-Kernel-Machine/venv/lib/python3.13/site-packages/numpy/_core/_methods.py:134: RuntimeWarning: overflow encountered in reduce
  ret = umr_su

DNN  train=116.5664  test=113.5004  epochs=890  time=37.7s  cpu_peak=2.3MB


In [18]:
# ResNet
def resnet_core():
    return run_resnet(train_x, train_y, test_x, test_y)

((res_result, res_gpu), res_cpu_delta, res_cpu_peak), res_time = measure_time(
    lambda: measure_memory_cpu(lambda: measure_memory_gpu(resnet_core)))
res_train_mse, res_test_mse, res_epochs, res_net, res_optimizer, \
    res_trainloss, res_testloss, res_device = res_result
print(f'ResNet  train={res_train_mse:.4f}  test={res_test_mse:.4f}  '
      f'epochs={res_epochs}  time={res_time:.1f}s  cpu_peak={res_cpu_peak:.1f}MB')

ValueError: Input contains NaN.

In [ ]:
# DKL
def dkl_core():
    return run_dkl(train_x, train_y, test_x, test_y)

((dkl_result, dkl_gpu), dkl_cpu_delta, dkl_cpu_peak), dkl_time = measure_time(
    lambda: measure_memory_cpu(lambda: measure_memory_gpu(dkl_core)))
dkl_train_mse, dkl_test_mse = dkl_result
print(f'DKL  train={dkl_train_mse:.4f}  test={dkl_test_mse:.4f}  '
      f'time={dkl_time:.1f}s  cpu_peak={dkl_cpu_peak:.1f}MB')

In [ ]:
# DGP
def dgp_core():
    return run_dgp(train_x, train_y, test_x, test_y)

((dgp_result, dgp_gpu), dgp_cpu_delta, dgp_cpu_peak), dgp_time = measure_time(
    lambda: measure_memory_cpu(lambda: measure_memory_gpu(dgp_core)))
dgp_train_mse, dgp_test_mse = dgp_result
print(f'DGP  train={dgp_train_mse:.4f}  test={dgp_test_mse:.4f}  '
      f'time={dgp_time:.1f}s  cpu_peak={dgp_cpu_peak:.1f}MB')

In [ ]:
# MLKM (default D=128 for first run; best D found in dimension experiment)
def mlkm_core():
    return run_mlkm(train_x, train_y, test_x, test_y, D1=128, D2=128, D3=128)

((mlkm_result, mlkm_gpu), mlkm_cpu_delta, mlkm_cpu_peak), mlkm_time = measure_time(
    lambda: measure_memory_cpu(lambda: measure_memory_gpu(mlkm_core)))
mlkm_train_mse, mlkm_test_mse, mlkm_epochs, mlkm_net, mlkm_optimizer, \
    mlkm_trainloss, mlkm_testloss, mlkm_device = mlkm_result
print(f'MLKM  train={mlkm_train_mse:.4f}  test={mlkm_test_mse:.4f}  '
      f'epochs={mlkm_epochs}  time={mlkm_time:.1f}s  cpu_peak={mlkm_cpu_peak:.1f}MB')

In [ ]:
# ResKernelNet (default D1=128)
def rkn_core():
    return run_reskernelnet(train_x, train_y, test_x, test_y, D1=128)

((rkn_result, rkn_gpu), rkn_cpu_delta, rkn_cpu_peak), rkn_time = measure_time(
    lambda: measure_memory_cpu(lambda: measure_memory_gpu(rkn_core)))

# FIX: correctly unpack all 8 return values (original unpacked only 5)
rkn_train_mse, rkn_test_mse, rkn_epochs, rkn_net, rkn_optimizer, \
    rkn_trainloss, rkn_testloss, rkn_device = rkn_result
print(f'ResKernelNet  train={rkn_train_mse:.4f}  test={rkn_test_mse:.4f}  '
      f'epochs={rkn_epochs}  time={rkn_time:.1f}s  cpu_peak={rkn_cpu_peak:.1f}MB')

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
results = pd.DataFrame({
    'Method':      ['RF+Ridge', 'DNN', 'ResNet', 'DKL', 'DGP', 'MLKM', 'ResKernelNet'],
    'Train MSE':   [rf_train_mse, dnn_train_mse, res_train_mse,
                    dkl_train_mse, dgp_train_mse, mlkm_train_mse, rkn_train_mse],
    'Test MSE':    [rf_test_mse,  dnn_test_mse,  res_test_mse,
                    dkl_test_mse,  dgp_test_mse,  mlkm_test_mse,  rkn_test_mse],
    'Time (s)':    [rf_time, dnn_time, res_time, dkl_time, dgp_time, mlkm_time, rkn_time],
    'CPU Peak (MB)':[rf_cpu_peak, dnn_cpu_peak, res_cpu_peak,
                     dkl_cpu_peak, dgp_cpu_peak, mlkm_cpu_peak, rkn_cpu_peak],
    'GPU Peak (MB)':[rf_gpu, dnn_gpu, res_gpu, dkl_gpu, dgp_gpu, mlkm_gpu, rkn_gpu],
})
print('\n── Method Comparison (sorted by Test MSE) ──')
print(results.sort_values('Test MSE').round(4).to_string(index=False))

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, tl, tel, color in [
    ('DNN',          dnn_trainloss,  dnn_testloss,  'blue'),
    ('ResNet',       res_trainloss,  res_testloss,  'green'),
    ('MLKM',         mlkm_trainloss, mlkm_testloss, 'red'),
    ('ResKernelNet', rkn_trainloss,  rkn_testloss,  'purple'),
]:
    axes[0].plot(tl,  label=name, color=color)
    axes[1].plot(tel, label=name, color=color)
axes[0].set_title('Train MSE'); axes[0].set_xlabel('Epoch')
axes[1].set_title('Test MSE');  axes[1].set_xlabel('Epoch')
for ax in axes:
    ax.legend(); ax.set_ylabel('MSE')
plt.tight_layout()
plt.savefig('msd_training_curves.png', dpi=150)
plt.show()

In [ ]:
# ── Conformal prediction (batched) ────────────────────────────────────────────
# FIX: use conformal_prediction_split_batch with correct signature
conformal_rows = []

for label, net, dev in [
    ('DNN',          dnn_net,  dnn_device),
    ('ResNet',       res_net,  res_device),
    ('MLKM',         mlkm_net, mlkm_device),
    ('ResKernelNet', rkn_net,  rkn_device),
]:
    cov, length, q = conformal_prediction_split_batch(
        net, dev, calibration_x, calibration_y, test_x, test_y, alpha=0.05)
    conformal_rows.append({'Method': label, 'Coverage': cov,
                           'Avg Interval Length': length, 'q_hat': q})
    print(f'{label:15s}  coverage={cov:.4f}  interval_len={length:.4f}  q_hat={q:.4f}')

conformal_df = pd.DataFrame(conformal_rows)
print('\n── Conformal Prediction Results (target coverage = 0.95) ──')
print(conformal_df.round(4).to_string(index=False))

In [ ]:
# ── Method comparison plots ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
methods = results['Method'].tolist()
colors  = plt.cm.tab10(np.linspace(0, 1, len(methods)))

for ax, col, title in zip(
    axes.flatten(),
    ['Test MSE', 'Train MSE', 'Time (s)', 'CPU Peak (MB)'],
    ['Test MSE', 'Train MSE', 'Runtime (s)', 'CPU Peak Memory (MB)']
):
    ax.bar(methods, results[col], color=colors)
    ax.set_title(title)
    ax.set_xticklabels(methods, rotation=30, ha='right')

plt.tight_layout()
plt.savefig('msd_method_comparison.png', dpi=150)
plt.show()

## Comprehensive Experiment: Hidden Dimension Configurations
Test all 7 methods across different hidden dimension settings.

In [ ]:
# ── Hidden dim configurations ─────────────────────────────────────────────────
hidden_configs = [
    (32,  64,  16),
    (64,  128, 32),
    (128, 256, 64),
    (256, 512, 128),
]
D_default = 128   # fixed D for MLKM/ResKernelNet in this sweep

hidden_dim_results = []

for h1, h2, h3 in hidden_configs:
    cfg = f'({h1},{h2},{h3})'
    print(f'\n── Config {cfg} ──')
    row = {'Config': cfg, 'h1': h1, 'h2': h2, 'h3': h3}

    # DNN
    r, t = measure_time(lambda: run_dnn(
        train_x, train_y, test_x, test_y, h1=h1, h2=h2, h3=h3))
    row['DNN_train'], row['DNN_test'], row['DNN_time'] = r[0], r[1], t
    print(f'  DNN    train={r[0]:.4f}  test={r[1]:.4f}')

    # ResNet
    r, t = measure_time(lambda: run_resnet(
        train_x, train_y, test_x, test_y, h1=h1, h2=h2, h3=h3))
    row['ResNet_train'], row['ResNet_test'], row['ResNet_time'] = r[0], r[1], t
    print(f'  ResNet train={r[0]:.4f}  test={r[1]:.4f}')

    # MLKM
    r, t = measure_time(lambda: run_mlkm(
        train_x, train_y, test_x, test_y,
        D1=D_default, D2=D_default, D3=D_default, h1=h1, h2=h2, h3=h3))
    row['MLKM_train'], row['MLKM_test'], row['MLKM_time'] = r[0], r[1], t
    print(f'  MLKM   train={r[0]:.4f}  test={r[1]:.4f}')

    # ResKernelNet
    r, t = measure_time(lambda: run_reskernelnet(
        train_x, train_y, test_x, test_y,
        D1=D_default, h1=h1, h2=h2, h3=h3))
    row['RKN_train'], row['RKN_test'], row['RKN_time'] = r[0], r[1], t
    print(f'  RKN    train={r[0]:.4f}  test={r[1]:.4f}')

    hidden_dim_results.append(row)

hidden_df = pd.DataFrame(hidden_dim_results)
hidden_df.to_csv('msd_hidden_dim_comparison.csv', index=False)
print('\n── Hidden dim results ──')
print(hidden_df.round(4).to_string(index=False))

In [ ]:
# ── Hidden dim plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
configs = hidden_df['Config'].tolist()
x = np.arange(len(configs))
w = 0.2

for ax, metric in zip(axes, ['test', 'train']):
    for i, (method, col_suffix) in enumerate(
        [('DNN','DNN'), ('ResNet','ResNet'), ('MLKM','MLKM'), ('RKN','RKN')]):
        ax.bar(x + i*w, hidden_df[f'{col_suffix}_{metric}'],
               width=w, label=method)
    ax.set_xticks(x + 1.5*w)
    ax.set_xticklabels(configs, rotation=20)
    ax.set_title(f'{metric.capitalize()} MSE by Hidden Dim Config')
    ax.set_ylabel('MSE'); ax.legend()

plt.tight_layout()
plt.savefig('msd_hidden_dim_comparison.png', dpi=150)
plt.show()

## MLKM RFF Dimension Experiment
Sweep D (= D1 = D2 = D3) for MLKM and ResKernelNet to find best performance.

In [ ]:
# ── D-dimension sweep ─────────────────────────────────────────────────────────
D_values = list(range(16, 321, 16))   # 16, 32, ..., 320
dim_results = []

best_mlkm_test   = float('inf'); best_mlkm_D   = None; best_mlkm_model   = None
best_rkn_test    = float('inf'); best_rkn_D    = None; best_rkn_model    = None

for D in tqdm(D_values, desc='D sweep'):
    # MLKM
    mlkm_r, mlkm_t = measure_time(
        lambda: run_mlkm(train_x, train_y, test_x, test_y, D1=D, D2=D, D3=D))
    mlkm_tr, mlkm_te, mlkm_ep = mlkm_r[0], mlkm_r[1], mlkm_r[2]
    if mlkm_te < best_mlkm_test:
        best_mlkm_test = mlkm_te; best_mlkm_D = D
        best_mlkm_model = (mlkm_r[3], mlkm_r[4], mlkm_r[5], mlkm_r[7])

    # ResKernelNet
    rkn_r, rkn_t = measure_time(
        lambda: run_reskernelnet(train_x, train_y, test_x, test_y, D1=D))
    rkn_tr, rkn_te, rkn_ep = rkn_r[0], rkn_r[1], rkn_r[2]
    if rkn_te < best_rkn_test:
        best_rkn_test = rkn_te; best_rkn_D = D
        best_rkn_model = (rkn_r[3], rkn_r[4], rkn_r[5], rkn_r[7])

    dim_results.append({
        'D': D,
        'MLKM_train': mlkm_tr, 'MLKM_test': mlkm_te,
        'MLKM_epochs': mlkm_ep, 'MLKM_time': mlkm_t,
        'RKN_train':  rkn_tr,  'RKN_test':  rkn_te,
        'RKN_epochs':  rkn_ep,  'RKN_time':  rkn_t,
    })

dim_df = pd.DataFrame(dim_results)
dim_df.to_csv('msd_mlkm_dimension_experiment_results.csv', index=False)

print(f'Best MLKM  D={best_mlkm_D}  test MSE={best_mlkm_test:.4f}')
print(f'Best RKN   D={best_rkn_D}   test MSE={best_rkn_test:.4f}')

In [ ]:
# ── D-sweep plots ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, method, train_col, test_col in [
    (axes[0,0], 'MLKM',         'MLKM_train', 'MLKM_test'),
    (axes[0,1], 'ResKernelNet', 'RKN_train',  'RKN_test'),
]:
    ax.plot(dim_df['D'], dim_df[train_col], label='Train MSE', marker='o', ms=4)
    ax.plot(dim_df['D'], dim_df[test_col],  label='Test MSE',  marker='s', ms=4)
    ax.set_title(f'{method}: MSE vs D')
    ax.set_xlabel('D (RFF dimension)'); ax.set_ylabel('MSE')
    ax.legend()

for ax, method, time_col, epoch_col in [
    (axes[1,0], 'MLKM',         'MLKM_time', 'MLKM_epochs'),
    (axes[1,1], 'ResKernelNet', 'RKN_time',  'RKN_epochs'),
]:
    ax2 = ax.twinx()
    ax.plot(dim_df['D'],  dim_df[time_col],  color='tab:blue',  marker='o', ms=4, label='Time (s)')
    ax2.plot(dim_df['D'], dim_df[epoch_col], color='tab:orange', marker='s', ms=4, label='Epochs')
    ax.set_title(f'{method}: Time & Epochs vs D')
    ax.set_xlabel('D'); ax.set_ylabel('Time (s)', color='tab:blue')
    ax2.set_ylabel('Epochs', color='tab:orange')
    ax.legend(loc='upper left'); ax2.legend(loc='upper right')

plt.tight_layout()
plt.savefig('msd_mlkm_dimension_plots.png', dpi=150)
plt.show()

In [ ]:
# ── Conformal prediction using best D models ──────────────────────────────────
# FIX: use conformal_prediction_split_batch with correct signature
print('\n── Conformal Prediction (best MLKM & ResKernelNet) ──')

if best_mlkm_model is not None:
    net_, _, _, dev_ = best_mlkm_model
    cov, length, q = conformal_prediction_split_batch(
        net_, dev_, calibration_x, calibration_y, test_x, test_y)
    print(f'MLKM (D={best_mlkm_D})  coverage={cov:.4f}  '
          f'interval_len={length:.4f}  q_hat={q:.4f}')

if best_rkn_model is not None:
    net_, _, _, dev_ = best_rkn_model
    cov, length, q = conformal_prediction_split_batch(
        net_, dev_, calibration_x, calibration_y, test_x, test_y)
    print(f'RKN  (D={best_rkn_D})   coverage={cov:.4f}  '
          f'interval_len={length:.4f}  q_hat={q:.4f}')

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('\n═══════════════════════════════════════════════════════════')
print('           MSD EXPERIMENT SUMMARY')
print('═══════════════════════════════════════════════════════════')
print(f'Dataset : Year Prediction MSD  (train={len(train_x)}, '
      f'calib={len(calibration_x)}, test={len(test_x)})')
print(f'Features: {INPUT_DIM}   Hidden: ({HIDDEN_DIM1}, {HIDDEN_DIM2}, {HIDDEN_DIM3})')
print()
print(results.sort_values('Test MSE').round(4).to_string(index=False))
print()
print('Conformal prediction (95% target):')
print(conformal_df.round(4).to_string(index=False))

In [ ]:
import sys
print(sys.executable)
print(sys.version)
